In [ ]:
!pip install transformers torch torchvision tqdm

!git clone https://github.com/seshuad/IMagenet.git

fatal: destination path 'IMagenet' already exists and is not an empty directory.


In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms, datasets
from transformers import ViTModel, ViTImageProcessor, SwinModel, AutoImageProcessor, CLIPModel, CLIPProcessor
from torch.utils.data import DataLoader
from tqdm import tqdm
import os
import pickle

# 1. Configuración
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATASET_PATH = "./IMagenet/tiny-imagenet-200/train/"
SAVE_PATH = "./precomputed_data/"

if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

# 2. Cargar Modelos
print(f"Cargando modelos en {DEVICE}...")
models = {
    "vit": ViTModel.from_pretrained('google/vit-base-patch16-224').to(DEVICE).eval(),
    "swin": SwinModel.from_pretrained("microsoft/swin-tiny-patch4-window7-224").to(DEVICE).eval(),
    "dino": ViTModel.from_pretrained("facebook/dino-vitb16").to(DEVICE).eval(),
    "clip": CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
}

# 3. Preparar Dataset (Batch size máis grande para GPU)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(root=DATASET_PATH, transform=transform)
loader = DataLoader(dataset, batch_size=128, num_workers=2, shuffle=False)

# Gardar rutas
image_paths = [s[0] for s in dataset.samples]
with open(os.path.join(SAVE_PATH, "image_paths.pkl"), "wb") as f:
    pickle.dump(image_paths, f)

# 4. Extracción
features = {"vit": [], "swin": [], "dino": [], "clip": []}

with torch.no_grad():
    for imgs, _ in tqdm(loader, desc="Extraendo"):
        imgs = imgs.to(DEVICE)

        # ViT & DINO
        features["vit"].append(F.normalize(models["vit"](imgs).last_hidden_state[:, 0, :], p=2, dim=1).cpu())
        features["dino"].append(F.normalize(models["dino"](imgs).last_hidden_state[:, 0, :], p=2, dim=1).cpu())

        # Swin
        out_swin = models["swin"](imgs).pooler_output
        features["swin"].append(F.normalize(out_swin, p=2, dim=1).cpu())

        # CLIP
        out_clip = models["clip"].get_image_features(pixel_values=imgs)
        if not isinstance(out_clip, torch.Tensor):
            out_clip = getattr(out_clip, "image_embeds", getattr(out_clip, "pooler_output", out_clip))
        features["clip"].append(F.normalize(out_clip, p=2, dim=1).cpu())

# 5. Gardar
for name in features:
    torch.save(torch.cat(features[name]), os.path.join(SAVE_PATH, f"features_{name}.pt"))
print("¡Feito!")

Cargando modelos en cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: facebook/dino-vitb16
Key                 | Status  | 
--------------------+---------+-
pooler.dense.bias   | MISSING | 
pooler.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
!zip -r precomputed_data.zip ./precomputed_data/
from google.colab import files
files.download('precomputed_data.zip')